Visualise model output from CF registry data vs baseline ppFEV1

In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import cfr.cfr_viz_helpers as vh

In [2]:
df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# EXCEL
# 2 entries means there is a 2019 entry for every 2023 entry
# df_res = bd.load_meas_from_excel(
#     "infer_AR_using_19_23_data_2entries_fev1_10122025",
#     # "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
#     study_folder="CFR",
#     str_cols_to_arrays=["Airway resistance (%)"],
# )
# df_res = df_res.drop(columns=["Healthy FEV1 (L)"])

# CSV
df_res = (
    bd.load_meas_from_excel(
        "infer_AR_using_two_days_model_19_23_data_2entries_fev1_10122025",
        # "infer_AR_using_two_days_model_19_23_data_2entries_fev1_fef2575_10122025",
        study_folder="CFR",
        str_cols_to_arrays=["Airway resistance (%)"],
        use_csv=True,
        date_cols=["Day"],
        bypass_sanity_checks=True,
    )
    .drop(columns=["Healthy FEV1 (L)"])
    .rename(columns={"Day": "Date Recorded"})
)

In [3]:
# Get AC from AR

AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df_res[AC.name] = df_res[AR.name].apply(lambda arr: arr[::-1])

# Check AC_mean = 100 - AR_mean
# ((df_res[AC.name].apply(lambda row: AC.get_mean(row)) - 100 + df_res[AR.name].apply(lambda row: AR.get_mean(row))) > 0.1).sum()

In [4]:
## Merging df on
df = df_res.merge(df_meas, on=["ID", "Date Recorded"])
# Keep only values from 2023
df23 = df[df["Date Recorded"] == datetime.date(2023, 1, 1)]
df23["clipped ecFEV1%Predicted"] = df23["ecFEV1 % Predicted"].clip(upper=100)

In [32]:
## FILL ##
title = f"Dumbell plot for CF Registry 2023, 2019 2nd day, FEV1 (2entries)"
# title = (
# f"Dumbell plot for CF Registry data 2023, 2019 2nd day, FEV1 & FEF25-75 (2entries)"
# )
# title = f"Dumbell plot for CF Registry data 2023, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 & FEF2575 (2entries)"

ac_col = AC.name

fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

ppfev1_row="clipped ecFEV1%Predicted"
# ppfev1_row = "ecFEV1 % Predicted"

df_to_plot, _, _ = vh.get_dumbell_plot_data(
    df23, AC.name, AC, ppfev1_row=ppfev1_row
)

# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title} (#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)}) clipped"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
vh.plot_dumbell_for_df(
    fig, df_mild, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 3
)
vh.plot_dumbell_for_df(
    fig, df_moderate, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 2
)
vh.plot_dumbell_for_df(
    fig, df_severe, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 1
)


fig.update_layout(
    height=1800,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    # f"{dh.get_path_to_main()}PlotsCFR/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf"
)
fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 952
# Moderate: 415
# Severe: 118
15.551501621761803


In [6]:
df23[df23.ID == 'B168645']

,ID,Date Recorded,Airway resistance (%),Airway conductance (%),Age,Height,FEV1,FEF2575,Sex,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted,idx ecFEV1 (L),idx ecFEF25-75 % ecFEV1 (%),clipped ecFEV1%Predicted
203,B168645,2023-01-01,"[1.83892842e-19, 2.10063922e-16, 1.427256e-13,...","[2.52441478e-283, 6.91532422e-247, 9.60520093e...",76,165,2.17,0.93,Female,2.17,0.93,42.857142,2.10407,103.13347,103.13347,43,21,100.0
